In [ ]:
import sys
print(sys.path)

In [ ]:
import os
sys.path.append(os.path.abspath(".."))

In [ ]:
import importlib
from resources import psiBfield

importlib.reload(psiBfield)

g = psiBfield.extract_g()
psi_norm, BR, Bphi, BZ = psiBfield.compute_B(g)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.optim as optim

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

In [ ]:
g.r_grid

In [ ]:
r = np.concatenate(g.r_grid)
print(r.shape)

In [ ]:
z = np.concatenate(g.z_grid)
X_train = np.column_stack((r, z))

In [ ]:
def plot(R ,Z, psi_preds, title):
  plt.contourf(R, Z, psi_preds,  cmap = 'plasma')
  plt.colorbar()
  plt.xlim([R[0],R[-1]])
  plt.ylim([Z[0],Z[-1]])
  plt.xlabel('R [m]')
  plt.ylabel('Z [m]')
  plt.axis('equal')
  plt.title(title)

In [ ]:
# NN
class BRNN(nn.Module):
  def __init__(self):
    super(BRNN, self).__init__()
    self.model = nn.Sequential(
        nn.Linear(2, 512),
        nn.ReLU(),
        nn.Linear(512, 512),
        nn.ReLU(),
        nn.Linear(512, 1)

    )
  def forward(self, x):
    return self.model(x)

In [ ]:
# Initialize model, loss, and optimizer
model_BR = BRNN()
fn_Loss = nn.MSELoss()
model_BR.to(device)
optimizer = optim.Adam(model_BR.parameters(), lr=0.001)

In [ ]:
BR_train  = np.concatenate(BR)

In [ ]:
BR_train = BR_train.reshape(-1,1)
print(BR_train.shape, X_train.shape)

In [ ]:
X_train.dtype, BR_train.dtype

In [ ]:
# Move BR_train to the device as well
BR_train = torch.tensor(BR_train, dtype=torch.float32).to(device)
# Move X_train to the device as well
X_train = torch.tensor(X_train, dtype=torch.float32).to(device)

# Training loop
epochs = 10001
for epoch in range(epochs):
  model_BR.train()
  BR_preds = model_BR(X_train)
  loss = fn_Loss(BR_preds, BR_train)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  if epoch % 100 == 0:
    print(f"Epoch {epoch}, Loss: {loss.item()}")

In [ ]:
# Testing



In [ ]:
# Compute final MSE loss
model_BR.eval()
with torch.no_grad():
  BR_final = model_BR(X_train)
  loss = fn_Loss(BR_final, BR_train)
  loss_BR = loss.item() * 100
  print(f"Final MSE Loss for (R, Z) -> BR (in %): {loss_BR}")

In [ ]:
BR_final = BR_final.reshape(129,129)

In [ ]:
BR_final.detach().numpy().dtype

In [ ]:
fig = plt.figure(figsize=(12, 4))

plt.subplot(1,3,1)
plot(g.r_grid[:, 0] ,g.z_grid[0, :], BR.T, title = r"$B_{R}(R,Z)_{Normalized}$")

plt.subplot(1,3,2)
plot(g.r_grid[:, 0] ,g.z_grid[0, :], BR_final.detach().numpy().T, title = r"$B_{R}(R,Z)_{predicted}$")

plt.subplot(1,3,3)
plot(g.r_grid[:, 0], g.z_grid[0, :], (abs(BR-BR_final.detach().numpy())*100).T, title=r"$B_{R}(R, Z)_{error}$")

plt.tight_layout()
plt.show()